# Patent Drawing Page Extractor

This notebook runs the same page-first extraction and cross-page assembly implementation as `page_extractor.py`. It consumes patent PDFs plus the CSV created by `page_classifier.py predict`, then creates one delivery folder for every PDF.

It preserves complete drawing pages, joins only confident cross-page plates, and marks uncertain relationships as `HOLD`. It does not crop individual figures.

## 1. Configuration

The defaults work with the package folders. Paths and run mode may also be supplied through the environment variables documented in `README.md`. The output folder must not already exist.

In [ ]:
from pathlib import Path
import json
import os
import random
import sys

def locate_package_dir():
    current = Path.cwd().resolve()
    candidates = [current, current / 'page_extractor']
    for candidate in candidates:
        if (candidate / 'page_extractor.py').is_file():
            return candidate
    raise FileNotFoundError('Could not locate page_extractor.py from the current directory')

PACKAGE_DIR = locate_package_dir()
PDF_INPUT = Path(os.environ.get('PAGE_EXTRACTOR_PDF_INPUT', PACKAGE_DIR / 'input_pdfs')).resolve()
PREDICTIONS = Path(os.environ.get('PAGE_EXTRACTOR_PREDICTIONS', PACKAGE_DIR / 'input_predictions' / 'full_predictions.csv')).resolve()
OUTPUT_ROOT = Path(os.environ.get('PAGE_EXTRACTOR_OUTPUT', PACKAGE_DIR / 'outputs' / 'notebook_run')).resolve()
RUN_MODE = os.environ.get('PAGE_EXTRACTOR_RUN_MODE', 'full').strip().lower()
SAMPLE_COUNT = int(os.environ.get('PAGE_EXTRACTOR_SAMPLE_COUNT', '30'))
RECURSIVE = os.environ.get('PAGE_EXTRACTOR_RECURSIVE', 'false').strip().lower() in {'1', 'true', 'yes'}
RANDOM_SEED = 42
RENDER_DPI = 150
WRITE_REVIEW_SHEETS = True

if RUN_MODE not in {'full', 'sample'}:
    raise ValueError("RUN_MODE must be 'full' or 'sample'")
if str(PACKAGE_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR))

from page_extractor import ALGORITHM_VERSION, find_pdf_inputs, load_prediction_table, extract_drawings

print(f'Algorithm:   {ALGORITHM_VERSION}')
print(f'PDF input:   {PDF_INPUT}')
print(f'Predictions: {PREDICTIONS}')
print(f'Output:      {OUTPUT_ROOT}')
print(f'Run mode:    {RUN_MODE}')

## 2. Validate and select inputs

`sample` mode takes a deterministic random sample. `full` mode processes every discovered PDF. The prediction table must contain a row for every page of every selected PDF.

In [ ]:
all_pdf_paths = find_pdf_inputs(PDF_INPUT, recursive=RECURSIVE)
predictions = load_prediction_table(PREDICTIONS)

if RUN_MODE == 'sample':
    randomizer = random.Random(RANDOM_SEED)
    pdf_paths = sorted(randomizer.sample(all_pdf_paths, min(SAMPLE_COUNT, len(all_pdf_paths))))
else:
    pdf_paths = all_pdf_paths

selected_names = {path.name for path in pdf_paths}
available_names = set(predictions['pdf'])
missing_names = sorted(selected_names - available_names)
if missing_names:
    raise ValueError(f'No classifier rows found for: {missing_names[:10]}')

print(f'Discovered PDFs: {len(all_pdf_paths)}')
print(f'Selected PDFs:   {len(pdf_paths)}')
print(f'Prediction rows: {len(predictions)}')

## 3. Extract complete drawing pages and assemble cross-page plates

In [ ]:
summary = extract_drawings(
    pdf_paths=pdf_paths,
    predictions=predictions,
    output_root=OUTPUT_ROOT,
    render_dpi=RENDER_DPI,
    write_review_sheets=WRITE_REVIEW_SHEETS,
)

## 4. Review the run summary and manifest

The final assertions inside the extractor have already checked exact-once drawing-page delivery, PDF-folder coverage, and the absence of TXT sidecars.

In [ ]:
import pandas as pd

display(pd.DataFrame([summary]).T.rename(columns={0: 'value'}))
manifest = pd.read_csv(OUTPUT_ROOT / 'metrics' / 'output_manifest.csv')
display(manifest.head(20))
print(f'Delivery folder: {OUTPUT_ROOT / "Extracted image"}')
print(f'Review sheets:   {OUTPUT_ROOT / "audit" / "reviews"}')